In [0]:
import pytest
from pyspoark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

@pytest.fixture(scope="session")
def spark():
    # Cria um ambiente Spark gratuito e osilado dentro do próprio GitHub Actions
    return SparkSession.builder \
        .master("Local[*]") \
        .appName("DataOps-Testing") \
        .getOrCreate()

def test_quarentena_registros_nuls(spark):
    # Cria uma massa de dados fake contendo um erro intencional (ID nulo)        
    schema = StructType([
        StructField("transaciont_id", StringType(), True),
        StructField("user_id", StringType(), True)
        ])
    dados_teste = [("TX_12345", "USR_1"), (None, "USR_2")] # o segundo registro deve ir pra quarentena
    df = spark.createDataFrame(dados_teste, schema)
    
    # Aplica a regra de validação
    df_trusted = df.filter(df.transacion_id.isNotNull())
    df_quarentena = df.filter(df.transacion_id.isNull())
    
    # Asserções
    assert df_trusted.count() == 1
    assert df_quarentena.count() == 1
    
    
    